# ScoutCard AI: run the tracker on a GPU

Runtime → Change runtime type → **T4 GPU** first. Then run the cells top to bottom.

Before this, on your Mac (in the repo folder), make the one file this notebook takes:

```bash
cd ~/scoutai && git pull origin claude/scoutcard-ai-v1-scaffold-c4bgpf
zip -r ~/Desktop/video-service.zip video-service \
  -x "video-service/.venv/*" "*/__pycache__/*" "video-service/tracks/*" \
     "video-service/detections/*" "video-service/debug/*" "video-service/recordings/*"
```

It holds the code, the cropped clips and each clip's calibration `.json`. The clips are game film, so use
your own Google account and delete the runtime when you're done (Runtime → Disconnect and delete runtime).

In [ ]:
!nvidia-smi -L
import torch
print("cuda:", torch.cuda.is_available())

## 1. Upload `video-service.zip` (drag it into the Files panel on the left, or run this cell)

In [ ]:
import os
if not os.path.exists("video-service.zip"):
    from google.colab import files
    files.upload()  # pick video-service.zip
!rm -rf video-service && unzip -q video-service.zip && ls video-service/clips

## 2. Install (the same OpenCV the Mac uses)

In [ ]:
!pip install -q "opencv-python-headless==5.0.0.93" transformers timm pillow pytest
import cv2; print(cv2.__version__)

## 3. Run the tracker
`--region`: hold the camera first, then detect only on the field, cut so a man is the same size to the model all play (measured: about 2.6x faster at detecting than whole-frame tiles, and no half-box ghosts). Prints a `time:` line per clip.

In [ ]:
CHECKPOINT = "PekingU/rtdetr_v2_r50vd"
CONFIDENCE = "0.3"
SECONDS = 8
CLIPS = "clips"   # every clip; or make a folder with only the ones you want
import os
os.environ["MODEL_CHECKPOINT"] = CHECKPOINT
os.environ["DETECT_CONFIDENCE"] = CONFIDENCE
%cd /content/video-service
!python -m eval.tracks {CLIPS}/ --out tracks --seconds {SECONDS} --region --debug debug/ --detections detections/

## 4. Download the results
Send `results.zip` back (the detections hold numbers, no film) and paste the printed `time:` lines.

In [ ]:
%cd /content/video-service
!zip -qr results.zip detections tracks debug
from google.colab import files
files.download("results.zip")